In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

In [2]:
City = "Gurgaon"

In [3]:
# headers :  defining a common browser's user agent string, this code is trying to mimic a real browser request to potentially avoid blocks or get the same content a real user would see.
headers = {
    'authority': 'www.99acres.com',
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'accept-language': 'en-US,en;q=0.9',
    'cache-control': 'no-cache',
    'dnt': '1',
    'pragma': 'no-cache',
    'referer': f'https://www.99acres.com/flats-in-{City}-ffid-page',
    'sec-ch-ua': '"Chromium";v="107", "Not;A=Brand";v="8"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-user': '?1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/527.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36',
}

In [4]:
project_dir = r"C:\Users\THARUN TEJA S\ML_PROJECT"

subdirectories = [
    "Data",
    f"Data/{City}",
    f"Data/{City}/Flats",
    f"Data/{City}/Societies",
    f"Data/{City}/Residential",
    f"Data/{City}/Independent House"
]

for subdir in subdirectories:
    dir_path = os.path.join(project_dir, subdir)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
        print(f"Created directory: {dir_path}")
    else:
        print(f"Directory already exists: {dir_path}")


Directory already exists: C:\Users\THARUN TEJA S\ML_PROJECT\Data
Created directory: C:\Users\THARUN TEJA S\ML_PROJECT\Data/Gurgaon
Created directory: C:\Users\THARUN TEJA S\ML_PROJECT\Data/Gurgaon/Flats
Created directory: C:\Users\THARUN TEJA S\ML_PROJECT\Data/Gurgaon/Societies
Created directory: C:\Users\THARUN TEJA S\ML_PROJECT\Data/Gurgaon/Residential
Created directory: C:\Users\THARUN TEJA S\ML_PROJECT\Data/Gurgaon/Independent House


In [5]:
# giving start and end page number 
start = int(input("Enter page number where you got error in last run.\nEnter page number to start from:"))

end = start+12

page_number = start
req = 0

flats = pd.DataFrame()

try:
    while page_number < end:
        i=1
        url = f'https://www.99acres.com/flats-in-{City}-ffid-page-{page_number}'
        page = requests.get(url, headers=headers)
        pageSoup = BeautifulSoup(page.content, 'html.parser')
        req+=1
        for soup in pageSoup.select_one('div[data-label="SEARCH"]').select('section[data-hydration-on-demand="true"]'):
            # Extracting property name and property subname
            try:
                property_name = soup.select_one('a.srpTuple__propertyName').text.strip()
                # Extract link
                link = soup.select_one('a.srpTuple__propertyName')['href']
                society = soup.select_one('#srp_tuple_society_heading').text.strip()
            except:
                continue
            # Details page
            page = requests.get(link, headers=headers)
            dpageSoup = BeautifulSoup(page.content, 'html.parser')
            req += 1
            try:
                #price Range
                price = dpageSoup.select_one('#pdPrice2').text.strip()
            except:
                price = ''
            # Area
            try:
                area = soup.select_one('#srp_tuple_price_per_unit_area').text.strip()
            except:
                area =''
            # Area with Type
            try:
                areaWithType = dpageSoup.select_one('#factArea').text.strip()
            except:
                areaWithType = ''
            
            # Configuration includes bedroom , bathroom , balcony , additoinalroom
            try:
                bedRoom = dpageSoup.select_one('#bedRoomNum').text.strip()
            except:
                bedRoom = ''
            try:
                bathroom = dpageSoup.select_one('#bathroomNum').text.strip()
            except:
                bathroom = ''
            try:
                balcony = dpageSoup.select_one('#balconyNum').text.strip()
            except:
                balcony = ''

            try:
                additionalRoom = dpageSoup.select_one('#additionalRooms').text.strip()
            except:
                additionalRoom = ''

            # Address
            try:
                address = dpageSoup.select_one('#address').text.strip()
            except:
                address = ''
            # Floor Number
            try:
                floorNum = dpageSoup.select_one('#floorNumLabel').text.strip()
            except:
                floorNum = ''
            # direction of facing
            try:
                facing = dpageSoup.select_one('#facingLabel').text.strip()
            except:
                facing = ''

            try:
                agePossession = dpageSoup.select_one('#agePossessionLbl').text.strip()
            except:
                agePossession = ''

            # Nearby Locations

            try:
                nearbyLocations = [i.text.strip() for i in dpageSoup.select_one('div.NearByLocation__tagWrap').select('span.NearByLocation__infoText')]
            except:
                nearbyLocations = ''

            # Descriptions
            try:
                description = dpageSoup.select_one('#description').text.strip()
            except:
                description = ''

            # Furnish Details
            try:
                furnishDetails = [i.text.strip() for i in dpageSoup.select_one('#FurnishDetails').select('li')]
            except:
                furnishDetails = ''

            # Features
            if furnishDetails:
                try:
                    features = [i.text.strip() for i in dpageSoup.select('#features')[1].select('li')]
                except:
                    features = ''
            else:
                try:
                    features = [i.text.strip() for i in dpageSoup.select('#features')[0].select('li')]
                except:
                    features = ''



            # Rating by Features
            try:
                rating = [i.text for i in dpageSoup.select_one('div.review__rightSide>div>ul>li>div').select('div.ratingByFeature__circleWrap')]
            except:
                rating = ''
            # print(top_f)

            try:
                # Property ID
                property_id = dpageSoup.select_one('#Prop_Id').text.strip()
            except:
                property_id = ''

            # creating a dictionary with the given variables
            property_data = {
            'property_name': property_name,
            'link': link,
            'society': society,
            'price': price,
            'area': area,
            'areaWithType': areaWithType,
            'bedRoom': bedRoom,
            'bathroom': bathroom,
            'balcony': balcony,
            'additionalRoom': additionalRoom,
            'address': address,
            'floorNum': floorNum,
            'facing': facing,
            'agePossession': agePossession,
            'nearbyLocations': nearbyLocations,
            'description': description,
            'furnishDetails': furnishDetails,
            'features': features,
            'rating': rating,
            'property_id': property_id
        }


            temp_df = pd.DataFrame.from_records([property_data])
            print(temp_df)
            flats = pd.concat([flats, temp_df], ignore_index=True)
            i += 1
            if req % 4==0:
                time.sleep(10)
            if req % 15 == 0:
                time.sleep(50)
        print(f'{pageNumber} -> {i}')
        pageNumber += 1

except AttributeError as e:
    print(e)
    print("----------------")
    print("""Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.\n
            You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.""")

    csv_file_path = os.path.join(
        project_dir,
        "Data",
        City,
        "Flats",
        f"flats_{City}_data-page-{start}-{page_number-1}.csv"
    )

    # Save the scraped data in csv format
    if os.path.isfile(csv_file_path):
        flats.to_csv(csv_file_path, mode="a", header=False, index=False)
    else:
        flats.to_csv(csv_file_path, mode="w", header=True, index=False)

    print(f"Data saved to: {csv_file_path}")

Enter page number where you got error in last run.
Enter page number to start from: 4


'NoneType' object has no attribute 'select'
----------------
Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.

            You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.
Data saved to: C:\Users\THARUN TEJA S\ML_PROJECT\Data\Gurgaon\Flats\flats_Gurgaon_data-page-4-3.csv


In [6]:

import os
import pandas as pd

def combine_csv_files(folder_path, combined_file_path):
    all_data = []

    for file_name in os.listdir(folder_path):

        # Skip non-CSV files and the output file itself
        if not file_name.endswith(".csv"):
            continue

        if file_name == os.path.basename(combined_file_path):
            continue

        file_path = os.path.join(folder_path, file_name)

        # Skip empty files
        if os.path.getsize(file_path) == 0:
            print(f"Skipping empty file: {file_name}")
            continue

        try:
            df = pd.read_csv(file_path)

            if df.empty:
                print(f"No data rows in: {file_name}")
                continue

            all_data.append(df)
            print(f"Read: {file_name} ({len(df)} rows)")

        except pd.errors.EmptyDataError:
            print(f"Skipping empty CSV: {file_name}")

    if not all_data:
        print("No valid CSV data found. Combined file not created.")
        return

    combined_data = pd.concat(all_data, ignore_index=True)
    combined_data.to_csv(combined_file_path, index=False)

    print(f"\nCombined file saved: {combined_file_path}")
    print(f"Total properties: {len(combined_data)}")


# Your Windows project directory
project_dir = r"C:\Users\THARUN TEJA S\ML_PROJECT"

folder_path = os.path.join(project_dir, "Data", City, "Flats")
combined_file_path = os.path.join(folder_path, "flats.csv")

combine_csv_files(folder_path, combined_file_path)


Skipping empty CSV: flats_Gurgaon_data-page-4-3.csv
No valid CSV data found. Combined file not created.
